#  Chargement, fusion et nettoyage des données brutes (LeBonCoin + BienIci).


In [6]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [7]:
spark = SparkSession.builder \
    .appName('02_nettoyage_immobilier') \
    .master('local[*]') \
    .config('spark.ui.enabled', 'false') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

In [8]:
print('SparkSession cree, version:', spark.version)

SparkSession cree, version: 4.1.1


## Etape 1 — Chargement des deux CSV bruts

In [10]:
RAW = '../data/raw'

In [11]:
df_lbc = spark.read.options(header='true', encoding='UTF-8', multiLine='true', escape='"').csv(RAW + '/leboncoin_raw.csv')

In [12]:
df_bienici = spark.read.options(header='true', encoding='UTF-8', multiLine='true', escape='"').csv(RAW + '/bienici_raw.csv')

In [13]:
print(f'LeBonCoin  : {df_lbc.count()} lignes')

LeBonCoin  : 3870 lignes


In [14]:
print(f'BienIci    : {df_bienici.count()} lignes')

BienIci    : 1908 lignes


- Pour le scrapping du site LeBonCoin nous avons 3870 lignes et 1908 lignes pour BienIci, pour un debut nous allons faire le traitement et tester differents modèles, puis on verra s'il faut rajouter encore plus de lignes.

In [15]:
df_lbc.printSchema()


root
 |-- source: string (nullable = true)
 |-- url_annonce: string (nullable = true)
 |-- type_bien: string (nullable = true)
 |-- prix: string (nullable = true)
 |-- surface: string (nullable = true)
 |-- nb_pieces: string (nullable = true)
 |-- nb_chambres: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- description: string (nullable = true)
 |-- date_scraping: string (nullable = true)



In [16]:
df_bienici.printSchema()


root
 |-- source: string (nullable = true)
 |-- url_annonce: string (nullable = true)
 |-- type_bien: string (nullable = true)
 |-- prix: string (nullable = true)
 |-- surface: string (nullable = true)
 |-- nb_pieces: string (nullable = true)
 |-- nb_chambres: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- description: string (nullable = true)
 |-- date_scraping: string (nullable = true)



- Pour les deux sites, nous avons le même schéma, ce qui va nous permettre de concaténer plus facilement les deux jeux de données pour n’en former qu’un seul. y
- Nous allons ensuite faire plus d’investigation pour repérer les problèmes dans les données et effectuer le nettoyage nécessaire avant de passer aux statistiques, aux visualisations et aux modèles.

## Etape 2 — Fusion des deux sources

In [17]:
df = df_lbc.union(df_bienici)

In [18]:
print(f'Total fusionne : {df.count()} lignes')

Total fusionne : 5778 lignes


## Etape 3 — Vérification et suppression des doublons

- Nous allons commençer par voir s'il y'a des URL en double.

In [19]:
avant_suppression = df.count()

In [21]:
df = df.dropDuplicates(['url_annonce'])
apres_suppression = df.count()

In [ ]:
print(f'Avant : {avant_suppression} | Apres : {apres_suppression} | Doublons supprimes : {avant_suppression - apres_suppression}')

Avant : 5778 | Apres : 5763 | Doublons supprimes : 15


Row(source='bienici', url_annonce='https://www.bienici.com/annonce/vente/bordeaux/appartement/1piece/ag060811-521071574?q=%2Frecherche%2Fachat%2Fbordeaux-33000%3Fpage%3D14', type_bien='appartement', prix='115000.0', surface='25.0', nb_pieces='1.0', nb_chambres=None, ville='Bordeaux', code_postal='33800.0', description="Bordeaux studio à vendre.\n\nBORDEAUX NANSOUTY - STUDIO AU CALME - CAVE - LUMINEUX - VUE DÉGAGÉE SANS VIS-A-VIS - PARKING POSSIBLE\n\nOptimhome vous propose ce grand studio de 25,41 m² loi Carrez, situé dans une résidence bien entretenue avec ascenseur.\n\nSans perte de place, il comprend une entrée avec rangements, une pièce de vie lumineuse, une grande cuisine équipée ainsi qu'un salle d'eau avec WC.\nVous disposerez également d'une cave saine au sous-sol.\n\nEnfin, côté stationnement, une place de PARKING extérieure dans la résidence est également proposée en sus du prix indiqué (15.000€).\n\nLe bien est VENDU LOUÉ, une locataire soigneuse est en place depuis 11 ans a